# Baseline 3 — Dass et al. (2021): Random Forest + SHAP for Self-Paced MOOC Dropout

**Reference:** S. Dass, K. Gary, J. Cunningham, *"Predicting Student Dropout in Self-Paced MOOC Course Using Random Forest Model"*, *Information* 12(11):476, 2021.  
**DOI:** 10.3390/info12110476

---

### What this notebook reproduces

Dass et al. focus on a **self-paced** STEM MOOC and use **per-day** progress features
(progress, days active, accuracy, speed, etc.) feeding a **Random Forest**. They report
Accuracy ≈ 87.5%, AUC ≈ 94.5%, F1 ≈ 87.5% on a single course.  
Their distinctive contribution is **explainability via SHAP**.

We adapt to MOOCCubeX:

- Use the *progress-style* features available in the base table:
  `time_spent`, `days_active`, `activity_density`, `avg_actions_per_active_day`,
  `accuracy`, `total_actions`, `session_count`, `course_pass_rate`.
- Scale the cumulative ones by `W/4` for window simulation.
- Both binary and 5-class labels.
- Add a **SHAP feature-importance** section to mirror Dass et al.'s explainability.


In [1]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)

MODEL_BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
WINDOWS = [1, 2, 4]
LABEL_ORDER_5 = ["A", "B", "C", "D", "E"]

print("Imports OK — reading from", MODEL_BASE_TABLE)


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 3, Finished, Available, Finished, False)

Imports OK — reading from dbo.tbl_graphgrued_weekly_active_base_ordered5


In [2]:
# ============================================================
# 1) LOAD MODEL BASE TABLE (V4 balanced)
# ============================================================
SAMPLE_FRACTION = 1.0
MAX_ROWS = 300_000

sdf = spark.table(MODEL_BASE_TABLE)
total_rows = sdf.count()
print(f"V4 cohort rows: {total_rows:,}")

if SAMPLE_FRACTION < 1.0:
    sdf = sdf.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

pdf = sdf.toPandas()
if len(pdf) > MAX_ROWS:
    g = pdf.groupby("risk_label_5_idx", group_keys=False)
    pdf = g.apply(lambda d: d.sample(
        n=max(1, int(len(d) * MAX_ROWS / len(pdf))),
        random_state=SEED)).reset_index(drop=True)

print(f"Working dataframe shape: {pdf.shape}")
print("\n5-class distribution after sampling:")
print(pdf["risk_label_5_idx"]
      .map({0:"A",1:"B",2:"C",3:"D",4:"E"})
      .value_counts(normalize=True).round(4)
      .reindex(LABEL_ORDER_5))


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 4, Finished, Available, Finished, False)

V4 cohort rows: 100,409
Working dataframe shape: (100409, 71)

5-class distribution after sampling:
risk_label_5_idx
A    0.0222
B    0.0812
C    0.0769
D    0.6411
E    0.1785
Name: proportion, dtype: float64


In [3]:
# ============================================================
# 2) LABELS — already in V4; just alias
# ============================================================
pdf["y_5class"] = pdf["risk_label_5_idx"].astype(int)
pdf["y_binary"] = pdf["is_completed"].fillna(0).astype(int)

print("y_5class distribution:")
counts5 = pdf["y_5class"].value_counts().sort_index()
print(counts5)
min_share = counts5.min() / counts5.sum()
if min_share < 0.005:
    raise RuntimeError(
        f"Class collapse: smallest class is {min_share:.2%} < 0.5%. "
        f"Re-run 01_v4_relabel_balanced.ipynb with adjusted thresholds.")

print("\ny_binary distribution:")
print(pdf["y_binary"].value_counts(normalize=True).round(4))


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 5, Finished, Available, Finished, False)

y_5class distribution:
y_5class
0     2231
1     8156
2     7721
3    64375
4    17926
Name: count, dtype: int64

y_binary distribution:
y_binary
0    0.6692
1    0.3308
Name: proportion, dtype: float64


In [4]:
# ============================================================
# 3) FEATURE GROUPS — TRUE weekly columns + static
# ============================================================
def _by_suffix(suffix):
    return sorted([c for c in pdf.columns if c.endswith(suffix)])

WEEK_COLS = {
    1: _by_suffix("_week1"),
    2: _by_suffix("_week1") + _by_suffix("_week2"),
    3: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3"),
    4: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3") + _by_suffix("_week4"),
}

# ĐÃ BỔ SUNG CÁC CỘT TEXT/METADATA TỪ BẢNG MỚI VÀO ĐÂY
EXCLUDE = {
    "user_id", "course_id", "enroll_time",
    "school_id", "course_name", "about", "prerequisites", "gender", "year_of_birth",
    "is_completed", "completion_rate",
    "risk_label_5_name", "risk_label_5_idx",
    "label_D_flag", "label_E_flag",
    "y_binary", "y_5class",
    "total_4w_interactions", "post_w1_interactions", "perf_composite",
    "course_norm_score",
    "problems_passed", "course_raw_score_sum", "total_problems",
    "risk_label_5_surrogate",
}
ALL_WEEKLY = set().union(*WEEK_COLS.values())

STATIC_COLS = [
    c for c in pdf.columns
    if c not in EXCLUDE
    and c not in ALL_WEEKLY
    and pdf[c].dtype.kind in "biufc"
]

print(f"Static feature count: {len(STATIC_COLS)}")
print("Static cols:", STATIC_COLS[:20], "..." if len(STATIC_COLS) > 20 else "")
print()
for W, cols in WEEK_COLS.items():
    print(f"W={W}: {len(cols)} weekly columns up to and including week {W}")

StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 6, Finished, Available, Finished, False)

Static feature count: 15
Static cols: ['num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_count_total_4w', 'video_watch_time_total_4w', 'chapter_coverage_ratio_raw', 'chapter_coverage_ratio', 'avg_chapter_progress_4w', 'chapter_progress_ratio_raw', 'chapter_progress_ratio', 'activity_signal_4w', 'is_active_4w'] 

W=1: 9 weekly columns up to and including week 1
W=2: 18 weekly columns up to and including week 2
W=3: 27 weekly columns up to and including week 3
W=4: 36 weekly columns up to and including week 4


In [5]:
# ============================================================
# 4) BUILD WINDOW VIEWS — leakage-safe (only weeks 1..W visible)
# ============================================================
def make_window_view(W):
    cols = STATIC_COLS + WEEK_COLS[W]
    cols = [c for c in cols if c in pdf.columns]
    Xdf  = pdf[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce").astype(float)
    return Xdf, cols

window_views = {}
for W in WINDOWS:
    Xdf, cols = make_window_view(W)
    window_views[W] = {"X": Xdf, "cols": cols}
    nan_pct = Xdf.isna().mean().mean()
    print(f"W={W}: shape={Xdf.shape}, mean NaN ratio={nan_pct:.3%}")


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 7, Finished, Available, Finished, False)

W=1: shape=(100409, 24), mean NaN ratio=24.351%
W=2: shape=(100409, 33), mean NaN ratio=41.389%
W=4: shape=(100409, 51), mean NaN ratio=59.149%


In [6]:
# ============================================================
# 5) STRATIFIED 8:1:1 SPLIT (per label)
# ============================================================
def split_idx(y, test_size=0.1, val_size=0.1, seed=SEED):
    idx = np.arange(len(y))
    idx_tv, idx_te = train_test_split(
        idx, test_size=test_size, stratify=y, random_state=seed)
    val_rel = val_size / (1.0 - test_size)
    idx_tr, idx_va = train_test_split(
        idx_tv, test_size=val_rel, stratify=y[idx_tv], random_state=seed)
    return idx_tr, idx_va, idx_te

splits = {}
for label in ["y_binary", "y_5class"]:
    y = pdf[label].values
    splits[label] = split_idx(y)
    n_tr, n_va, n_te = (len(splits[label][0]), len(splits[label][1]), len(splits[label][2]))
    print(f"{label}: train={n_tr}, val={n_va}, test={n_te}")


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 8, Finished, Available, Finished, False)

y_binary: train=80327, val=10041, test=10041
y_5class: train=80327, val=10041, test=10041


In [7]:
# ============================================================
# 6) METRIC HELPERS — EarlyRisk-MOOC metric set
# ============================================================
def evaluate_classifier(y_true, y_pred, y_proba=None, classes=None, name=""):
    res = {"name": name}
    res["accuracy"]    = accuracy_score(y_true, y_pred)
    res["macro_f1"]    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    res["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if classes is None:
        classes = sorted(list(set(y_true) | set(y_pred)))
    per_class_f1 = f1_score(y_true, y_pred, average=None,
                            labels=classes, zero_division=0)
    for cls, f in zip(classes, per_class_f1):
        res[f"f1_{cls}"] = float(f)
    if y_proba is not None and len(classes) >= 2:
        try:
            if len(classes) == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr",
                    labels=classes, average="macro")
        except Exception as e:
            res["auc_error"] = str(e)
    return res


def flatten_result(res, name, classes_order, runtime, W, label_col):
    out = {"name": name, "label": label_col, "window": W,
           "runtime_sec": round(runtime, 2)}
    for k, v in res.items():
        if k.startswith("f1_") and k != "f1_score":
            try:
                idx = int(k.split("_")[1])
                out[f"f1_{classes_order[idx]}"] = v
            except (ValueError, IndexError):
                out[k] = v
        elif k in ("accuracy","macro_f1","weighted_f1","auc","auc_ovr"):
            out[k] = v
    return out


print("Metric helpers ready")


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 9, Finished, Available, Finished, False)

Metric helpers ready


In [8]:
# ============================================================
# 7) IMPUTATION + SCALING (fit on TRAIN only)
# ============================================================
def fit_impute_scale(X_tr, X_te, X_va=None):
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr_i = imp.transform(X_tr)
    X_te_i = imp.transform(X_te)
    sc = StandardScaler().fit(X_tr_i)
    out = (sc.transform(X_tr_i), sc.transform(X_te_i))
    if X_va is not None:
        out = out + (sc.transform(imp.transform(X_va)),)
    return out

print("fit_impute_scale ready")


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 10, Finished, Available, Finished, False)

fit_impute_scale ready


In [9]:
# ============================================================
# 8) DASS — RandomForest grid + SHAP
# ============================================================
from sklearn.ensemble import RandomForestClassifier
SHAP_AVAILABLE = False
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    pass


all_results = []
trained = {}
for label_col, classes in [("y_binary", [0,1]), ("y_5class", LABEL_ORDER_5)]:
    idx_tr, idx_va, idx_te = splits[label_col]
    y_all = pdf[label_col].values
    n_cls = len(classes)
    for W in WINDOWS:
        Xdf = window_views[W]["X"]
        X = Xdf.values
        X_tr_s, X_te_s = fit_impute_scale(X[idx_tr], X[idx_te])
        rf = RandomForestClassifier(n_estimators=300, min_samples_split=5,
                                    min_samples_leaf=2, random_state=SEED, n_jobs=-1)
        print(f"[{label_col}] W={W} RF ...")
        t0 = time.time(); rf.fit(X_tr_s, y_all[idx_tr]); rt = time.time()-t0
        y_pred = rf.predict(X_te_s); y_proba = rf.predict_proba(X_te_s)
        res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                   classes=list(range(n_cls)), name="RandomForest")
        row = flatten_result(res, "RandomForest", classes, rt, W, label_col)
        all_results.append(row)
        trained[(label_col, W)] = (rf, X_tr_s, X_te_s, y_all[idx_tr], y_all[idx_te], list(Xdf.columns))
        print(f"   acc={row.get('accuracy',0):.4f}  macroF1={row.get('macro_f1',0):.4f}  rt={row['runtime_sec']}s")


# Feature importance on the W=4 binary model (Dass-style explanation)
if ("y_binary", 4) in trained:
    rf, Xtr, Xte, ytr, yte, feat_names = trained[("y_binary", 4)]
    if SHAP_AVAILABLE:
        bg = Xtr[np.random.RandomState(SEED).choice(len(Xtr),
                  size=min(500, len(Xtr)), replace=False)]
        explainer = shap.TreeExplainer(rf, bg)
        sv = explainer.shap_values(Xte[:500], check_additivity=False)
        if isinstance(sv, list): sv = sv[1]
        importance = np.abs(sv).mean(axis=0)
        method = "SHAP |mean|"
    else:
        importance = rf.feature_importances_
        method = "RF feature_importances_"
    imp_df = (pd.DataFrame({"feature": feat_names, "importance": importance})
                .sort_values("importance", ascending=False).head(20))
    print(f"\nTop-20 features ({method}) — RF on W=4 binary:")
    print(imp_df.round(4).to_string(index=False))


StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 11, Finished, Available, Finished, False)

[y_binary] W=1 RF ...


   acc=0.9217  macroF1=0.9096  rt=24.83s
[y_binary] W=2 RF ...


   acc=0.9252  macroF1=0.9136  rt=18.97s
[y_binary] W=4 RF ...


   acc=0.9263  macroF1=0.9151  rt=21.45s
[y_5class] W=1 RF ...


   acc=0.9114  macroF1=0.8446  rt=18.02s
[y_5class] W=2 RF ...


   acc=0.9166  macroF1=0.8515  rt=20.44s
[y_5class] W=4 RF ...


 99%|===================| 986/1000 [04:04<00:03]       


Top-20 features (SHAP |mean|) — RF on W=4 binary:
                    feature  importance
         activity_signal_4w      0.0672
      correct_answers_week1      0.0540
     num_active_enrollments      0.0511
  completed_questions_week1      0.0470
              num_resources      0.0258
      correct_answers_week2      0.0233
  completed_questions_week2      0.0207
         course_max_chapter      0.0175
      correct_answers_week4      0.0168
        course_total_videos      0.0167
problem_attempt_count_week1      0.0151
  completed_questions_week4      0.0146
          total_score_week1      0.0141
      correct_answers_week3      0.0126
  completed_questions_week3      0.0107
problem_attempt_count_week4      0.0106
          total_score_week2      0.0088
problem_attempt_count_week2      0.0086
problem_attempt_count_week3      0.0075
          total_score_week4      0.0046


In [10]:
res_df = pd.DataFrame(all_results)
print("\n=== Dass et al. 2021 — V4 benchmark ===")
print(res_df.round(4).to_string(index=False))
res_df.to_csv("/lakehouse/default/Files/tmp/results_dass_2021.csv", index=False)
print("\nSaved -> /lakehouse/default/Files/tmp/results_dass_2021.csv")

StatementMeta(, 9c14922b-d7bb-4452-8d84-f014ba51e58d, 12, Finished, Available, Finished, True)


=== Dass et al. 2021 — V4 benchmark ===
        name    label  window  runtime_sec  accuracy  macro_f1  weighted_f1   f1_0   f1_1    auc   f1_A   f1_B   f1_C   f1_D   f1_E  auc_ovr
RandomForest y_binary       1        24.83    0.9217    0.9096       0.9208 0.9427 0.8764 0.9713    NaN    NaN    NaN    NaN    NaN      NaN
RandomForest y_binary       2        18.97    0.9252    0.9136       0.9243 0.9452 0.8820 0.9729    NaN    NaN    NaN    NaN    NaN      NaN
RandomForest y_binary       4        21.45    0.9263    0.9151       0.9255 0.9459 0.8843 0.9728    NaN    NaN    NaN    NaN    NaN      NaN
RandomForest y_5class       1        18.02    0.9114    0.8446       0.9093    NaN    NaN    NaN 0.8029 0.8384 0.7312 0.9442 0.9063   0.9851
RandomForest y_5class       2        20.44    0.9166    0.8515       0.9143    NaN    NaN    NaN 0.8137 0.8402 0.7373 0.9470 0.9190   0.9854
RandomForest y_5class       4        22.77    0.9210    0.8562       0.9187    NaN    NaN    NaN 0.8166 0.8394 0.